# 01: Regularized Regression (Ridge, Lasso, ElasticNet) & Regularization Paths

**Track 04: Regression & Continuous Prediction** | *Tensorbox AI/ML Production Curriculum*

---
### Overview & Objectives
Master L1/L2 regularization mathematics, shrink collinear coefficients, construct alpha paths, and evaluate out-of-sample Generalization Error (RMSE, MAE, R²).


## 1. Mathematical Formulation of Regularized Loss Functions

**Ordinary Least Squares (OLS):**
$$\mathcal{L}_{OLS}(w) = \frac{1}{2n} \sum_{i=1}^n (y_i - x_i^T w)^2$$

**Ridge Regression ($L_2$ Penalty):**
$$\mathcal{L}_{Ridge}(w) = \mathcal{L}_{OLS}(w) + \alpha \|w\|_2^2 = \frac{1}{2n} \|y - Xw\|_2^2 + \alpha \sum_{j=1}^p w_j^2$$

**Lasso Regression ($L_1$ Penalty - Sparse Feature Selection):**
$$\mathcal{L}_{Lasso}(w) = \frac{1}{2n} \|y - Xw\|_2^2 + \alpha \sum_{j=1}^p |w_j|$$

**ElasticNet (Convex Combination):**
$$\mathcal{L}_{ElasticNet}(w) = \frac{1}{2n} \|y - Xw\|_2^2 + \alpha \left(\rho \|w\|_1 + \frac{1-\rho}{2} \|w\|_2^2\right)$$

In [ ]:
import os
import sys
from pathlib import Path

for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (p / "utils").exists():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        break

from utils.data_loader import load_dataset

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, ElasticNetCV
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

df = load_dataset("housing_prices", split="train")
numeric_df = df.select_dtypes(include=[np.number]).dropna()

if "Price" in df.columns:
    y = df["Price"]
    X = numeric_df.drop(columns=["Price", "Id"], errors="ignore")
elif "medv" in df.columns:
    y = df["medv"]
    X = numeric_df.drop(columns=["medv"], errors="ignore")
else:
    y = numeric_df.iloc[:, -1]
    X = numeric_df.iloc[:, :-1]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train.shape}, Test set: {X_test.shape}")


## 2. Model Fitting & Cross-Validated Optimal Alpha Search
Evaluate performance across OLS, Ridge, Lasso, and ElasticNet.

In [ ]:
models = {
    "OLS Linear": LinearRegression(),
    "Ridge (L2)": RidgeCV(alphas=np.logspace(-3, 3, 50), cv=5),
    "Lasso (L1)": LassoCV(alphas=np.logspace(-3, 3, 50), cv=5, random_state=42),
    "ElasticNet": ElasticNetCV(l1_ratio=[0.1, 0.5, 0.7, 0.9, 0.99], cv=5, random_state=42)
}

results = []
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    
    best_alpha = getattr(model, "alpha_", "N/A")
    results.append({"Model": name, "RMSE": round(rmse, 2), "MAE": round(mae, 2), "R2 Score": round(r2, 4), "Optimal Alpha": best_alpha})

results_df = pd.DataFrame(results)
print("=== Regularized Model Performance Comparison ===")
print(results_df.to_string(index=False))

## 3. Coefficient Sparsity & Feature Attribution
Inspect the weights assigned by Lasso and Ridge.

In [ ]:
lasso_coefs = pd.Series(models["Lasso (L1)"].coef_, index=X.columns)
print("Lasso Coefficients (L1 Zeroed Features):")
print(lasso_coefs)
